# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

In [2]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("-", "_")
)

In [3]:
for col in ["total_claim_amount", "monthly_premium_auto", "customer_lifetime_value"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["response", "policy_type", "gender", "state", "education"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()


In [4]:
low_claim_threshold = 1000

df_low_yes = df.loc[
    (df["total_claim_amount"] < low_claim_threshold) &
    (df["response"].str.lower() == "yes")
].copy()

print("Original rows:", df.shape[0])
print("Filtered (low claims + yes) rows:", df_low_yes.shape[0])

Original rows: 10910
Filtered (low claims + yes) rows: 1399


2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [5]:
df_yes = df[df["response"].str.lower() == "yes"].copy()


In [6]:
premium_clv_analysis = (
    df_yes
    .groupby(["policy_type", "gender"])
    .agg(
        avg_monthly_premium=("monthly_premium_auto", "mean"),
        avg_clv=("customer_lifetime_value", "mean"),
        customer_count=("customer", "count")
    )
    .round(2)
    .reset_index()
)

premium_clv_analysis


,policy_type,gender,avg_monthly_premium,avg_clv,customer_count
0,Corporate Auto,F,94.30,7712.63,169
1,Corporate Auto,M,92.19,7944.47,154
2,Personal Auto,F,99.00,8339.79,540
3,Personal Auto,M,91.09,7448.38,536
4,Special Auto,F,92.31,7691.58,35
5,Special Auto,M,86.34,8247.09,32


In [7]:
claims_analysis = (
    df_yes
    .groupby(["policy_type", "gender"])
    .agg(
        avg_claim_amount=("total_claim_amount", "mean"),
        median_claim_amount=("total_claim_amount", "median")
    )
    .round(2)
    .reset_index()
)

claims_analysis


,policy_type,gender,avg_claim_amount,median_claim_amount
0,Corporate Auto,F,433.74,420.04
1,Corporate Auto,M,408.58,364.80
2,Personal Auto,F,452.97,424.33
3,Personal Auto,M,457.01,412.80
4,Special Auto,F,453.28,420.04
5,Special Auto,M,429.53,345.60


In [8]:
segment_summary = premium_clv_analysis.merge(
    claims_analysis,
    on=["policy_type", "gender"]
)

segment_summary


,policy_type,gender,avg_monthly_premium,avg_clv,customer_count,avg_claim_amount,median_claim_amount
0,Corporate Auto,F,94.30,7712.63,169,433.74,420.04
1,Corporate Auto,M,92.19,7944.47,154,408.58,364.80
2,Personal Auto,F,99.00,8339.79,540,452.97,424.33
3,Personal Auto,M,91.09,7448.38,536,457.01,412.80
4,Special Auto,F,92.31,7691.58,35,453.28,420.04
5,Special Auto,M,86.34,8247.09,32,429.53,345.60


3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [9]:
customers_by_state = df["state"].value_counts()
customers_by_state

state
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
nan            631
Name: count, dtype: int64

In [10]:
states_over_500 = customers_by_state[customers_by_state > 500]
states_over_500


state
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
nan            631
Name: count, dtype: int64

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [11]:
clv_by_edu_gender = (
    df
    .groupby(["education", "gender"])["customer_lifetime_value"]
    .agg(
        min_clv="min",
        median_clv="median",
        max_clv="max",
        customer_count="count"
    )
    .round(2)
    .reset_index()
    .sort_values("median_clv", ascending=False)
)

clv_by_edu_gender


,education,gender,min_clv,median_clv,max_clv,customer_count
7,High School or Below,M,1940.98,6286.73,83325.38,1556
6,High School or Below,F,2144.92,6039.55,55277.45,1590
3,College,M,1918.12,6005.85,61134.68,1596
8,Master,F,2417.78,5729.86,51016.07,475
0,Bachelor,F,1904.00,5640.51,73225.96,1700
2,College,F,1898.68,5623.61,61850.19,1608
9,Master,M,2272.31,5579.10,50568.26,412
5,Doctor,M,2267.60,5577.67,32677.34,201
1,Bachelor,M,1898.01,5548.03,67907.27,1572
4,Doctor,F,2395.57,5332.46,44856.11,200


## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9